# Random Forest Meta-Model

Full energy-ticker Random Forest comparison notebook. It mirrors the logistic notebook output structure: CPCV ranking by AUC, stitched CPCV path Sharpe diagnostics, per-ticker final OOS evaluation, and universal comparison CSVs.

In [1]:
from __future__ import annotations

from pathlib import Path
from itertools import combinations
import copy
import json
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

RANDOM_STATE = 42


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "features").exists() and (candidate / "03_model_development").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with data/features and 03_model_development.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_SELECTION_DIR = PROJECT_ROOT / "03_model_development" / "Feature Selection"
if str(FEATURE_SELECTION_DIR) not in sys.path:
    sys.path.insert(0, str(FEATURE_SELECTION_DIR))

from feature_selection_methods import (  # noqa: E402
    CorrelationClusterResult,
    CorrelationClusterSelector,
    DEFAULT_EXCLUDE_COLUMNS,
    PCAFeatureReducer,
    PCAFeatureReductionResult,
    infer_numeric_feature_columns,
    run_correlation_cluster_selection,
    run_pca_feature_reduction,
)

warnings.filterwarnings("ignore")

print("Project root:", PROJECT_ROOT)
print("Feature-selection utilities imported from:", FEATURE_SELECTION_DIR)


Project root: /Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw
Feature-selection utilities imported from: /Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/03_model_development/Feature Selection


## 1. Settings

In [2]:
TICKERS = ["cl1s", "ho1s", "rb1s", "ng1s"]
TICKER = TICKERS[0]  # Used for preview cells only.

RUN_FULL_GRID = True
SMOKE_CONFIG_NAMES = ["ewma_10d_tp2_sl2"]

TEST_START_DATE = "2022-01-01"
TEST_END_DATE = "2022-06-30"

DATE_COL = "date"
INSTRUMENT_COL = "instrument"
TARGET_COL = "metalabel"
TRAIN_ON_NONZERO_SIGNALS_ONLY = True

# Match the reduced CPCV setting used for the logistic search.
CPCV_N_BLOCKS = 8
CPCV_N_TEST_BLOCKS = 2

print("Tickers:", TICKERS)
print("Run full grid:", RUN_FULL_GRID)
print("CPCV:", CPCV_N_BLOCKS, "blocks,", CPCV_N_TEST_BLOCKS, "test blocks")


Tickers: ['cl1s', 'ho1s', 'rb1s', 'ng1s']
Run full grid: True
CPCV: 8 blocks, 2 test blocks


## 2. Paths And Non-Feature Columns

In [3]:
CLEAN_FEATURE_DIR = PROJECT_ROOT / "data" / "features" / "clean_feature_set"
TRIPLE_BARRIER_DIR = PROJECT_ROOT / "data" / "features" / "triple_barrier"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "data" / "model_outputs" / "random_forest"
RESULTS_DIR = PROJECT_ROOT / "results"
RF_RESULTS_PATH = RESULTS_DIR / "random_forest.csv"
MODEL_COMPARISON_RESULTS_PATH = RESULTS_DIR / "model_comparison.csv"

NON_FEATURE_COLS = {
    DATE_COL,
    INSTRUMENT_COL,
    TARGET_COL,
    "target",
    "label",
    "y",
    "meta_label",
    "tb_label",
    "barrier_label",
    "signal_column",
    "training_end",
    "num_days",
    "t1",
    "timeout_date",
    "timeout_close",
    "touch_date",
    "touch_price",
    "touched_barrier",
    "vertical_barrier_date",
    "barrier_touch_date",
    "event_end_date",
    "exit_date",
    "exit_price",
    "triple_barrier_label",
    "holding_period_days",
    "raw_touch_return",
    "signed_touch_return",
    "exit_return",
    "triple_barrier_return",
    "tb_return",
    "realised_return",
    "realized_return",
    "pnl",
    "profit",
    "barrier_touched",
    "first_barrier_touched",
    "hit_barrier",
    "pt",
    "sl",
    "tp",
    "pt_mult",
    "sl_mult",
    "take_profit_mult",
    "stop_loss_mult",
    "vol",
    "volatility",
    "target_vol",
    "volatility_method",
    "ewma_span",
    "volatility_window",
    "close_tb",
}

PROTECTED_FEATURES = ["primary_signal"]

print("Clean feature dir:", CLEAN_FEATURE_DIR)
print("Triple-barrier dir:", TRIPLE_BARRIER_DIR)
print("Output dir:", MODEL_OUTPUT_DIR)


Clean feature dir: /Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/features/clean_feature_set
Triple-barrier dir: /Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/features/triple_barrier
Output dir: /Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest


## 3. Load Clean Features And Triple-Barrier Labels

In [4]:
def extract_num_days_from_filename(path: Path) -> int | None:
    filename = path.stem.lower()
    for pattern in [r"num_days[_\-]?(\d+)", r"(\d+)d", r"horizon[_\-]?(\d+)"]:
        match = re.search(pattern, filename)
        if match:
            return int(match.group(1))
    return None


def clean_feature_path_for_ticker(ticker: str) -> Path:
    return CLEAN_FEATURE_DIR / f"{ticker}_clean_feature_set.csv"


def load_feature_set(path: Path, ticker: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Feature set not found: {path}")
    features = pd.read_csv(path, parse_dates=[DATE_COL])
    if INSTRUMENT_COL in features.columns:
        features[INSTRUMENT_COL] = features[INSTRUMENT_COL].str.lower()
        features = features[features[INSTRUMENT_COL] == ticker.lower()].copy()
    features = features.sort_values(DATE_COL).reset_index(drop=True)
    if features.empty:
        raise ValueError(f"No clean feature rows found for {ticker}.")
    return features


def load_triple_barrier_files(triple_barrier_dir: Path, ticker: str) -> dict[str, pd.DataFrame]:
    if not triple_barrier_dir.exists():
        raise FileNotFoundError(f"Triple-barrier directory not found: {triple_barrier_dir}")

    csv_files = sorted(
        path for path in triple_barrier_dir.glob("*.csv")
        if ticker.lower() in path.name.lower() and path.name != "triple_barrier_config_summary.csv"
    )
    if not csv_files:
        raise FileNotFoundError(f"No triple-barrier CSV files found for {ticker} in {triple_barrier_dir}")

    selected_files = csv_files
    if not RUN_FULL_GRID:
        selected_files = [path for path in csv_files if path.stem.replace(f"{ticker.lower()}_", "") in SMOKE_CONFIG_NAMES]
        if not selected_files:
            raise ValueError(f"Smoke configs {SMOKE_CONFIG_NAMES} were not found for {ticker}.")

    tb_data = {}
    for path in selected_files:
        header = pd.read_csv(path, nrows=0).columns
        date_cols = [col for col in [DATE_COL, "training_end", "timeout_date", "touch_date"] if col in header]
        df = pd.read_csv(path, parse_dates=date_cols)
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])

        if INSTRUMENT_COL in df.columns:
            df[INSTRUMENT_COL] = df[INSTRUMENT_COL].str.lower()
            df = df[df[INSTRUMENT_COL] == ticker.lower()].copy()

        if "num_days" in df.columns:
            num_days_values = pd.to_numeric(df["num_days"], errors="coerce").dropna()
            if num_days_values.empty:
                raise ValueError(f"num_days column is empty/non-numeric in {path.name}")
            num_days = int(num_days_values.mode().iloc[0])
        else:
            num_days = extract_num_days_from_filename(path)
            if num_days is None:
                raise ValueError(f"Could not infer num_days for {path.name}.")
            df["num_days"] = num_days

        for required in [TARGET_COL, "timeout_date", "signed_touch_return"]:
            if required not in df.columns:
                raise ValueError(f"{required} not found in {path.name}")

        keep_cols = [DATE_COL]
        if INSTRUMENT_COL in df.columns:
            keep_cols.append(INSTRUMENT_COL)
        for col in [
            TARGET_COL,
            "primary_signal",
            "num_days",
            "timeout_date",
            "touch_date",
            "signed_touch_return",
            "raw_touch_return",
            "triple_barrier_label",
            "touched_barrier",
            "holding_period_days",
        ]:
            if col in df.columns and col not in keep_cols:
                keep_cols.append(col)

        df = df[keep_cols].copy()
        df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
        df["num_days"] = int(num_days)
        df["timeout_date"] = pd.to_datetime(df["timeout_date"])
        if "touch_date" in df.columns:
            df["touch_date"] = pd.to_datetime(df["touch_date"])
        df["signed_touch_return"] = pd.to_numeric(df["signed_touch_return"], errors="coerce")
        df = df.sort_values(DATE_COL).reset_index(drop=True)

        config_name = path.stem.replace(f"{ticker.lower()}_", "")
        tb_data[config_name] = df

    return tb_data


features_by_ticker = {}
tb_configs_by_ticker = {}
load_summary_rows = []

for ticker in TICKERS:
    features_by_ticker[ticker] = load_feature_set(clean_feature_path_for_ticker(ticker), ticker)
    tb_configs_by_ticker[ticker] = load_triple_barrier_files(TRIPLE_BARRIER_DIR, ticker)
    features_df = features_by_ticker[ticker]

    print("=" * 80)
    print(f"Loaded feature set for {ticker}: {features_df.shape}")
    print(f"Loaded {len(tb_configs_by_ticker[ticker])} triple-barrier configurations")

    for config_name, tb_df in tb_configs_by_ticker[ticker].items():
        load_summary_rows.append({
            "ticker": ticker,
            "config_name": config_name,
            "label_rows": len(tb_df),
            "label_start": tb_df[DATE_COL].min(),
            "label_end": tb_df[DATE_COL].max(),
            "num_days": int(tb_df["num_days"].mode().iloc[0]),
            "metalabel_0": int((tb_df[TARGET_COL] == 0).sum()),
            "metalabel_1": int((tb_df[TARGET_COL] == 1).sum()),
            "metalabel_1_rate": float((tb_df[TARGET_COL] == 1).mean()),
        })

load_summary_df = pd.DataFrame(load_summary_rows)
display(load_summary_df)
display(features_by_ticker[TICKER].head())
first_config_name = list(tb_configs_by_ticker[TICKER].keys())[0]
display(tb_configs_by_ticker[TICKER][first_config_name].head())


Loaded feature set for cl1s: (626, 147)
Loaded 6 triple-barrier configurations
Loaded feature set for ho1s: (628, 147)
Loaded 6 triple-barrier configurations
Loaded feature set for rb1s: (628, 147)
Loaded 6 triple-barrier configurations
Loaded feature set for ng1s: (628, 147)
Loaded 6 triple-barrier configurations


,ticker,config_name,label_rows,label_start,label_end,num_days,metalabel_0,metalabel_1,metalabel_1_rate
0,cl1s,ewma_10d_tp1_5_sl1_5,412,2020-01-07,2022-06-15,10,128,284,0.689320
1,cl1s,ewma_10d_tp2_sl2,412,2020-01-07,2022-06-15,10,127,285,0.691748
2,cl1s,ewma_5d_tp2_sl2,417,2020-01-07,2022-06-23,5,145,272,0.652278
3,cl1s,garman_klass_10d_tp2_sl2,412,2020-01-07,2022-06-15,10,131,281,0.682039
4,cl1s,parkinson_10d_tp2_sl2,412,2020-01-07,2022-06-15,10,131,281,0.682039
5,cl1s,rolling_10d_tp2_sl2,412,2020-01-07,2022-06-15,10,129,283,0.686893
6,ho1s,ewma_10d_tp1_5_sl1_5,63,2020-01-21,2022-06-02,10,23,40,0.634921
7,ho1s,ewma_10d_tp2_sl2,63,2020-01-21,2022-06-02,10,22,41,0.650794
8,ho1s,ewma_5d_tp2_sl2,63,2020-01-21,2022-06-02,5,22,41,0.650794
9,ho1s,garman_klass_10d_tp2_sl2,63,2020-01-21,2022-06-02,10,21,42,0.666667


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,...,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
0,2020-01-03,cl1s,0,24.795579,25.974970,24.775315,25.553469,2.185752e+06,958523.501762,0.030566,...,0.0,1.0,0.0,0.0,0.000000,0.022015,0.009606,0.000000,0.000000,0.000000
1,2020-01-06,cl1s,0,25.820960,26.230302,25.387301,25.642633,1.786962e+06,909240.146872,0.003489,...,0.0,0.0,0.0,1.0,0.000000,0.024615,0.009457,0.000000,0.000000,0.000000
2,2020-01-07,cl1s,-1,25.496729,25.593998,25.172498,25.411618,1.437614e+06,877280.234192,-0.009009,...,0.0,0.0,0.0,1.0,-0.999224,0.016524,0.009817,-0.000372,-0.000423,-0.000404
3,2020-01-08,cl1s,0,25.468359,26.607221,23.972842,24.159275,2.974939e+06,792303.827743,-0.049282,...,1.0,0.0,0.0,0.0,0.000000,-0.023747,0.015258,0.000000,0.000000,0.000000
4,2020-01-09,cl1s,0,24.313285,24.442978,23.774251,24.139011,1.852834e+06,695693.746602,-0.000839,...,0.0,0.0,0.0,1.0,0.000000,-0.025944,0.014936,0.000000,0.000000,0.000000


,date,instrument,metalabel,primary_signal,num_days,timeout_date,touch_date,signed_touch_return,raw_touch_return,triple_barrier_label,touched_barrier,holding_period_days
0,2020-01-07,cl1s,1,-1,10,2020-01-22,2020-01-08,0.049282,-0.049282,-1,lower,1
1,2020-01-22,cl1s,1,-1,10,2020-02-05,2020-01-24,0.044942,-0.044942,-1,lower,2
2,2020-01-23,cl1s,1,-1,10,2020-02-06,2020-01-27,0.044073,-0.044073,-1,lower,4
3,2020-01-24,cl1s,1,-1,10,2020-02-07,2020-01-30,0.037830,-0.037830,-1,lower,6
4,2020-01-27,cl1s,1,-1,10,2020-02-10,2020-01-31,0.029733,-0.029733,-1,lower,4


## 4. Build Modelling Tables

In [5]:
def get_feature_cols(df: pd.DataFrame, non_feature_cols: set[str]) -> list[str]:
    candidate_cols = [col for col in df.columns if col not in non_feature_cols]
    numeric_cols = [col for col in candidate_cols if pd.api.types.is_numeric_dtype(df[col])]
    dropped_non_numeric = sorted(set(candidate_cols) - set(numeric_cols))
    if dropped_non_numeric:
        print(f"Dropped non-numeric feature columns: {dropped_non_numeric}")
    return numeric_cols


def dataset_key(ticker: str, config_name: str) -> str:
    return f"{ticker}__{config_name}"


def merge_features_with_tb(features_df: pd.DataFrame, tb_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    features_clean = features_df.copy()
    tb_clean = tb_df.copy()
    features_clean[DATE_COL] = pd.to_datetime(features_clean[DATE_COL])
    tb_clean[DATE_COL] = pd.to_datetime(tb_clean[DATE_COL])

    if INSTRUMENT_COL in features_clean.columns:
        features_clean[INSTRUMENT_COL] = features_clean[INSTRUMENT_COL].str.lower()
        features_clean = features_clean[features_clean[INSTRUMENT_COL] == ticker.lower()].copy()
    if INSTRUMENT_COL in tb_clean.columns:
        tb_clean[INSTRUMENT_COL] = tb_clean[INSTRUMENT_COL].str.lower()
        tb_clean = tb_clean[tb_clean[INSTRUMENT_COL] == ticker.lower()].copy()

    tb_clean = tb_clean.drop(columns=["primary_signal"], errors="ignore")

    merge_keys = [DATE_COL]
    if INSTRUMENT_COL in features_clean.columns and INSTRUMENT_COL in tb_clean.columns:
        merge_keys.append(INSTRUMENT_COL)

    merged = features_clean.merge(tb_clean, on=merge_keys, how="inner", suffixes=("", "_tb"))
    return merged.sort_values(DATE_COL).reset_index(drop=True)


def clean_model_dataframe(merged_df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    df = merged_df.copy()
    required_metadata = [TARGET_COL, "num_days", "timeout_date", "signed_touch_return"]
    missing_metadata = [col for col in required_metadata if col not in df.columns]
    if missing_metadata:
        raise ValueError(f"Required metadata columns missing after merge: {missing_metadata}")

    if TRAIN_ON_NONZERO_SIGNALS_ONLY and "primary_signal" in df.columns:
        df = df[df["primary_signal"] != 0].copy()

    df = df.replace([np.inf, -np.inf], np.nan)
    required_cols = feature_cols + required_metadata + [DATE_COL]
    df = df.dropna(subset=required_cols).copy()

    df[TARGET_COL] = df[TARGET_COL].astype(int)
    invalid_targets = sorted(set(df[TARGET_COL].unique()) - {0, 1})
    if invalid_targets:
        raise ValueError(f"{TARGET_COL} must be binary 0/1. Found: {invalid_targets}")

    df["num_days"] = pd.to_numeric(df["num_days"], errors="coerce").astype("Int64")
    if df["num_days"].isna().any() or df["num_days"].nunique() != 1:
        raise ValueError("Each modelling dataset must have exactly one non-missing num_days value.")
    df["timeout_date"] = pd.to_datetime(df["timeout_date"])
    df["signed_touch_return"] = pd.to_numeric(df["signed_touch_return"], errors="coerce")
    return df.sort_values(DATE_COL).reset_index(drop=True)


def chronological_train_test_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    test_start = pd.Timestamp(TEST_START_DATE)
    test_end = pd.Timestamp(TEST_END_DATE)
    train_df = df[df[DATE_COL] < test_start].copy()
    test_df = df[(df[DATE_COL] >= test_start) & (df[DATE_COL] <= test_end)].copy()
    return train_df, test_df


datasets_by_config = {}
dataset_summary_rows = []

for ticker in TICKERS:
    features = features_by_ticker[ticker]
    tb_configs = tb_configs_by_ticker[ticker]
    for config_name, tb_df in tb_configs.items():
        print("=" * 80)
        print(f"Building dataset for {ticker} / triple-barrier config: {config_name}")
        merged_df = merge_features_with_tb(features, tb_df, ticker)
        feature_cols = get_feature_cols(merged_df, NON_FEATURE_COLS)
        assert "primary_signal" in feature_cols, "primary_signal should remain a model feature."
        assert not set(feature_cols).intersection(NON_FEATURE_COLS), "Non-feature columns leaked into feature_cols."

        model_df = clean_model_dataframe(merged_df, feature_cols)
        train_df, test_df = chronological_train_test_split(model_df)
        if train_df[TARGET_COL].nunique() < 2:
            raise ValueError(f"Training data for {ticker}/{config_name} needs both metalabel classes.")

        key = dataset_key(ticker, config_name)
        datasets_by_config[key] = {
            "ticker": ticker,
            "config_name": config_name,
            "merged_df": merged_df,
            "model_df": model_df,
            "train_df": train_df,
            "test_df": test_df,
            "feature_cols": feature_cols,
            "num_days": int(model_df["num_days"].mode().iloc[0]),
        }
        dataset_summary_rows.append({
            "dataset_key": key,
            "ticker": ticker,
            "config_name": config_name,
            "merged_rows": len(merged_df),
            "model_rows": len(model_df),
            "train_rows": len(train_df),
            "oos_rows": len(test_df),
            "feature_count": len(feature_cols),
            "num_days": datasets_by_config[key]["num_days"],
            "train_metalabel_1_rate": train_df[TARGET_COL].mean(),
            "oos_metalabel_1_rate": test_df[TARGET_COL].mean() if len(test_df) else np.nan,
        })

        print(f"Train rows: {len(train_df)} | OOS rows: {len(test_df)} | Features: {len(feature_cols)} | num_days: {datasets_by_config[key]['num_days']}")

dataset_summary_df = pd.DataFrame(dataset_summary_rows)
display(dataset_summary_df)
first_dataset_key = list(datasets_by_config.keys())[0]
data = datasets_by_config[first_dataset_key]
display(data["train_df"].head())
display(data["test_df"].head())


Building dataset for cl1s / triple-barrier config: ewma_10d_tp1_5_sl1_5
Train rows: 333 | OOS rows: 78 | Features: 145 | num_days: 10
Building dataset for cl1s / triple-barrier config: ewma_10d_tp2_sl2
Train rows: 333 | OOS rows: 78 | Features: 145 | num_days: 10
Building dataset for cl1s / triple-barrier config: ewma_5d_tp2_sl2
Train rows: 333 | OOS rows: 83 | Features: 145 | num_days: 5
Building dataset for cl1s / triple-barrier config: garman_klass_10d_tp2_sl2
Train rows: 333 | OOS rows: 78 | Features: 145 | num_days: 10
Building dataset for cl1s / triple-barrier config: parkinson_10d_tp2_sl2
Train rows: 333 | OOS rows: 78 | Features: 145 | num_days: 10
Building dataset for cl1s / triple-barrier config: rolling_10d_tp2_sl2
Train rows: 333 | OOS rows: 78 | Features: 145 | num_days: 10
Building dataset for ho1s / triple-barrier config: ewma_10d_tp1_5_sl1_5
Train rows: 61 | OOS rows: 2 | Features: 145 | num_days: 10
Building dataset for ho1s / triple-barrier config: ewma_10d_tp2_sl2
Tr

,dataset_key,ticker,config_name,merged_rows,model_rows,train_rows,oos_rows,feature_count,num_days,train_metalabel_1_rate,oos_metalabel_1_rate
0,cl1s__ewma_10d_tp1_5_sl1_5,cl1s,ewma_10d_tp1_5_sl1_5,411,411,333,78,145,10,0.660661,0.820513
1,cl1s__ewma_10d_tp2_sl2,cl1s,ewma_10d_tp2_sl2,411,411,333,78,145,10,0.663664,0.820513
2,cl1s__ewma_5d_tp2_sl2,cl1s,ewma_5d_tp2_sl2,416,416,333,83,145,5,0.612613,0.819277
3,cl1s__garman_klass_10d_tp2_sl2,cl1s,garman_klass_10d_tp2_sl2,411,411,333,78,145,10,0.654655,0.807692
4,cl1s__parkinson_10d_tp2_sl2,cl1s,parkinson_10d_tp2_sl2,411,411,333,78,145,10,0.654655,0.807692
5,cl1s__rolling_10d_tp2_sl2,cl1s,rolling_10d_tp2_sl2,411,411,333,78,145,10,0.660661,0.807692
6,ho1s__ewma_10d_tp1_5_sl1_5,ho1s,ewma_10d_tp1_5_sl1_5,63,63,61,2,145,10,0.655738,0.000000
7,ho1s__ewma_10d_tp2_sl2,ho1s,ewma_10d_tp2_sl2,63,63,61,2,145,10,0.672131,0.000000
8,ho1s__ewma_5d_tp2_sl2,ho1s,ewma_5d_tp2_sl2,63,63,61,2,145,5,0.672131,0.000000
9,ho1s__garman_klass_10d_tp2_sl2,ho1s,garman_klass_10d_tp2_sl2,63,63,61,2,145,10,0.688525,0.000000


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,...,signal_x_hmm_prob_positive_or_strong_upside,metalabel,num_days,timeout_date,touch_date,signed_touch_return,raw_touch_return,triple_barrier_label,touched_barrier,holding_period_days
0,2020-01-07,cl1s,-1,25.496729,25.593998,25.172498,25.411618,1.437614e+06,8.772802e+05,-0.009009,...,-0.000404,1,10,2020-01-22,2020-01-08,0.049282,-0.049282,-1,lower,1
1,2020-01-22,cl1s,-1,23.599977,23.648586,22.696648,22.984255,1.530855e+06,1.194576e+06,-0.028092,...,-0.007740,1,10,2020-02-05,2020-01-24,0.044942,-0.044942,-1,lower,2
2,2020-01-23,cl1s,-1,22.729054,22.793867,22.186246,22.518412,1.737915e+06,1.183091e+06,-0.020268,...,-0.003801,1,10,2020-02-06,2020-01-27,0.044073,-0.044073,-1,lower,4
3,2020-01-24,cl1s,-1,22.558920,22.664241,21.813573,21.951300,1.447108e+06,1.203433e+06,-0.025184,...,-0.011184,1,10,2020-02-07,2020-01-30,0.037830,-0.037830,-1,lower,6
4,2020-01-27,cl1s,-1,21.752811,21.756861,21.116834,21.525966,1.759849e+06,1.191300e+06,-0.019376,...,-0.004789,1,10,2020-02-10,2020-01-31,0.029733,-0.029733,-1,lower,4


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,...,signal_x_hmm_prob_positive_or_strong_upside,metalabel,num_days,timeout_date,touch_date,signed_touch_return,raw_touch_return,triple_barrier_label,touched_barrier,holding_period_days
333,2022-01-03,cl1s,1,22.564638,22.794190,22.141309,22.680904,1.065440e+06,918132.919438,0.011568,...,0.006656,1,10,2022-01-18,2022-01-06,0.044427,0.044427,1,upper,3
334,2022-01-04,cl1s,1,22.663017,23.145970,22.567619,22.952193,1.250668e+06,902998.030950,0.011961,...,0.004041,1,10,2022-01-19,2022-01-11,0.054942,0.054942,1,upper,7
335,2022-01-05,cl1s,1,23.008836,23.426202,22.809096,23.208575,1.334695e+06,882194.268008,0.011170,...,0.002001,1,10,2022-01-20,2022-01-11,0.043288,0.043288,1,upper,6
336,2022-01-06,cl1s,1,23.026723,23.921080,22.874682,23.688547,1.598908e+06,863986.782478,0.020681,...,0.970264,1,10,2022-01-21,2022-01-12,0.040020,0.040020,1,upper,6
337,2022-01-07,cl1s,1,23.736246,23.989648,23.408315,23.521600,1.390162e+06,801562.076196,-0.007048,...,0.001528,1,10,2022-01-24,2022-01-12,0.047402,0.047402,1,upper,5


## 5. Random Forest And Feature-Processing Configs

In [6]:
RF_CONFIGS = [
    {
        "model_name": "rf_shallow",
        "model_params": {
            "n_estimators": 300,
            "max_depth": 4,
            "min_samples_leaf": 10,
            "min_samples_split": 20,
            "max_features": "sqrt",
            "class_weight": "balanced",
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
    {
        "model_name": "rf_medium_depth",
        "model_params": {
            "n_estimators": 500,
            "max_depth": 8,
            "min_samples_leaf": 8,
            "min_samples_split": 20,
            "max_features": "sqrt",
            "class_weight": "balanced",
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
]

FEATURE_PROCESSING_CONFIGS = [
    {
        "feature_method": "corr_cluster",
        "feature_params": {
            "corr_threshold": 0.90,
            "corr_method": "spearman",
            "selection_method": "target_corr",
            "min_periods": 20,
        },
    },
    {
        "feature_method": "pca",
        "feature_params": {
            "n_components": 0.95,
            "standardize": True,
            "impute_strategy": "median",
            "component_prefix": "pca",
            "random_state": RANDOM_STATE,
            "svd_solver": "full",
        },
    },
]


def make_random_forest(model_params: dict) -> RandomForestClassifier:
    return RandomForestClassifier(**model_params)


display(pd.DataFrame([{"model_name": cfg["model_name"], **cfg["model_params"]} for cfg in RF_CONFIGS]))


,model_name,n_estimators,max_depth,min_samples_leaf,min_samples_split,max_features,class_weight,bootstrap,random_state,n_jobs
0,rf_shallow,300,4,10,20,sqrt,balanced,True,42,-1
1,rf_medium_depth,500,8,8,20,sqrt,balanced,True,42,-1


## 6. CPCV Splits

In [7]:
def make_time_blocks(n_samples: int, n_blocks: int) -> list[np.ndarray]:
    if n_samples < n_blocks:
        raise ValueError(f"Not enough samples for CPCV: n_samples={n_samples}, n_blocks={n_blocks}")
    return [block.astype(int) for block in np.array_split(np.arange(n_samples), n_blocks) if len(block) > 0]


def generate_cpcv_splits(
    train_df: pd.DataFrame,
    n_blocks: int = CPCV_N_BLOCKS,
    n_test_blocks: int = CPCV_N_TEST_BLOCKS,
    embargo_size: int | None = None,
    date_col: str = DATE_COL,
):
    df = train_df.sort_values(date_col).reset_index(drop=True).copy()
    if "timeout_date" not in df.columns:
        raise ValueError("timeout_date is required for purging.")
    if embargo_size is None:
        if "num_days" not in df.columns:
            raise ValueError("embargo_size was not provided and num_days is missing.")
        embargo_size = int(pd.to_numeric(df["num_days"], errors="coerce").dropna().mode().iloc[0])
    embargo_size = int(embargo_size)

    n_samples = len(df)
    blocks = make_time_blocks(n_samples, n_blocks)
    event_start = pd.to_datetime(df[date_col]).reset_index(drop=True)
    event_end = pd.to_datetime(df["timeout_date"]).reset_index(drop=True)

    for split_id, test_block_ids in enumerate(combinations(range(len(blocks)), n_test_blocks), start=1):
        test_idx = np.sort(np.concatenate([blocks[block_id] for block_id in test_block_ids]))
        test_block_by_index = {
            int(row_idx): int(block_id)
            for block_id in test_block_ids
            for row_idx in blocks[block_id]
        }
        test_mask = np.zeros(n_samples, dtype=bool)
        test_mask[test_idx] = True
        train_mask = ~test_mask

        for block_id in test_block_ids:
            block_idx = blocks[block_id]
            test_block_start = event_start.iloc[block_idx].min()
            test_block_end = event_end.iloc[block_idx].max()
            overlap_mask = ((event_start <= test_block_end) & (event_end >= test_block_start)).to_numpy()
            train_mask[overlap_mask] = False

            block_last_pos = int(block_idx.max())
            embargo_start_pos = block_last_pos + 1
            embargo_end_pos = min(block_last_pos + embargo_size, n_samples - 1)
            if embargo_start_pos <= embargo_end_pos:
                train_mask[np.arange(embargo_start_pos, embargo_end_pos + 1)] = False

        train_idx = np.where(train_mask)[0]
        assert set(train_idx).isdisjoint(set(test_idx)), "Train/test overlap found."

        yield {
            "split_id": split_id,
            "test_block_ids": tuple(int(block_id) for block_id in test_block_ids),
            "test_block_by_index": test_block_by_index,
            "train_idx": train_idx,
            "test_idx": test_idx,
            "embargo_size": embargo_size,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
        }


example_splits = list(generate_cpcv_splits(data["train_df"], embargo_size=data["num_days"]))
print("Example CPCV folds:", len(example_splits))
print("Example embargo size:", sorted({split["embargo_size"] for split in example_splits}))
print("Example train/test rows:", example_splits[0]["n_train"], example_splits[0]["n_test"])


Example CPCV folds: 28
Example embargo size: [10]
Example train/test rows: 239 84


## 7. Metrics And Fold-Safe Feature Processing

In [8]:
def get_positive_class_proba(model, X: pd.DataFrame) -> np.ndarray:
    if len(X) == 0:
        raise ValueError("Prediction input is empty.")
    proba = model.predict_proba(X)
    classes = list(model.classes_)
    if 1 not in classes:
        raise ValueError(f"Model was not trained with positive class 1. Classes: {classes}")
    return proba[:, classes.index(1)]


def per_trade_sharpe(returns: pd.Series) -> float:
    r = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < 2:
        return np.nan
    std = r.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan
    return float(r.mean() / std)


def annualised_sharpe(returns: pd.Series, periods_per_year: float = 252) -> float:
    r = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < 2:
        return np.nan
    std = r.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan
    return float(np.sqrt(periods_per_year) * r.mean() / std)


def max_drawdown(nav: pd.Series) -> float:
    nav = pd.Series(nav).dropna()
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def classification_metrics(y_true: pd.Series, y_proba: np.ndarray, threshold: float = 0.5) -> dict:
    y_true = pd.Series(y_true).astype(int)
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_proba) if y_true.nunique() == 2 else np.nan,
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics.update({"tn": tn, "fp": fp, "fn": fn, "tp": tp})
    return metrics


def split_protected_and_reducible_features(feature_cols: list[str]) -> tuple[list[str], list[str]]:
    protected = [col for col in PROTECTED_FEATURES if col in feature_cols]
    reducible = [col for col in feature_cols if col not in protected]
    return protected, reducible


def apply_feature_processing_for_fold(
    fold_train: pd.DataFrame,
    fold_test: pd.DataFrame,
    feature_cols: list[str],
    y_fold_train: pd.Series,
    feature_processing_config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    method = feature_processing_config["feature_method"]
    params = copy.deepcopy(feature_processing_config.get("feature_params", {}))

    protected_cols, reducible_cols = split_protected_and_reducible_features(feature_cols)
    X_train_protected = fold_train[protected_cols].reset_index(drop=True) if protected_cols else pd.DataFrame(index=range(len(fold_train)))
    X_test_protected = fold_test[protected_cols].reset_index(drop=True) if protected_cols else pd.DataFrame(index=range(len(fold_test)))

    if method == "none":
        return (
            fold_train[feature_cols].reset_index(drop=True).copy(),
            fold_test[feature_cols].reset_index(drop=True).copy(),
            feature_cols.copy(),
            {
                "feature_method": method,
                "n_original_features": len(feature_cols),
                "n_processed_features": len(feature_cols),
                "n_dropped_features": 0,
                "n_pca_components": np.nan,
                "pca_explained_variance": np.nan,
                "processor": None,
            },
        )

    if method == "corr_cluster":
        selector = CorrelationClusterSelector(**params)
        selector.fit(fold_train[reducible_cols], y=y_fold_train, feature_columns=reducible_cols)
        X_train_selected = selector.transform(fold_train[reducible_cols]).reset_index(drop=True)
        X_test_selected = selector.transform(fold_test[reducible_cols]).reset_index(drop=True)
        X_train_processed = pd.concat([X_train_protected, X_train_selected], axis=1)
        X_test_processed = pd.concat([X_test_protected, X_test_selected], axis=1)
        processed_cols = protected_cols + selector.selected_features_
        return X_train_processed, X_test_processed, processed_cols, {
            "feature_method": method,
            "n_original_features": len(feature_cols),
            "n_reducible_features": len(reducible_cols),
            "n_protected_features": len(protected_cols),
            "n_processed_features": len(processed_cols),
            "n_dropped_features": len(selector.dropped_features_),
            "n_pca_components": np.nan,
            "pca_explained_variance": np.nan,
            "processor": selector,
        }

    if method == "pca":
        reducer = PCAFeatureReducer(**params)
        reducer.fit(fold_train[reducible_cols], feature_columns=reducible_cols)
        X_train_pca = reducer.transform(fold_train[reducible_cols]).reset_index(drop=True)
        X_test_pca = reducer.transform(fold_test[reducible_cols]).reset_index(drop=True)
        X_train_processed = pd.concat([X_train_protected, X_train_pca], axis=1)
        X_test_processed = pd.concat([X_test_protected, X_test_pca], axis=1)
        processed_cols = protected_cols + reducer.component_columns_
        return X_train_processed, X_test_processed, processed_cols, {
            "feature_method": method,
            "n_original_features": len(feature_cols),
            "n_reducible_features": len(reducible_cols),
            "n_protected_features": len(protected_cols),
            "n_processed_features": len(processed_cols),
            "n_dropped_features": 0,
            "n_pca_components": len(reducer.component_columns_),
            "pca_explained_variance": float(reducer.cumulative_explained_variance_.iloc[-1]),
            "processor": reducer,
        }

    raise ValueError(f"Unknown feature processing method: {method}")


## 8. Run CPCV Grid

In [9]:
def run_cpcv_for_rf_config(
    rf_config: dict,
    train_df: pd.DataFrame,
    feature_cols: list[str],
    feature_processing_config: dict,
    n_blocks: int = CPCV_N_BLOCKS,
    n_test_blocks: int = CPCV_N_TEST_BLOCKS,
    embargo_size: int | None = None,
    threshold: float = 0.5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = train_df.sort_values(DATE_COL).reset_index(drop=True).copy()
    splits = list(generate_cpcv_splits(df, n_blocks=n_blocks, n_test_blocks=n_test_blocks, embargo_size=embargo_size))

    fold_rows = []
    prediction_rows = []

    for split in splits:
        fold_train = df.iloc[split["train_idx"]].copy()
        fold_test = df.iloc[split["test_idx"]].copy()
        y_train = fold_train[TARGET_COL].astype(int)
        y_test = fold_test[TARGET_COL].astype(int)
        if y_train.nunique() < 2:
            continue

        X_train_processed, X_test_processed, processed_cols, processing_info = apply_feature_processing_for_fold(
            fold_train=fold_train,
            fold_test=fold_test,
            feature_cols=feature_cols,
            y_fold_train=y_train,
            feature_processing_config=feature_processing_config,
        )

        model = make_random_forest(rf_config["model_params"])
        model.fit(X_train_processed, y_train)
        y_proba = get_positive_class_proba(model, X_test_processed)
        y_pred = (y_proba >= threshold).astype(int)
        accepted_returns = fold_test.loc[y_pred == 1, "signed_touch_return"]
        metrics = classification_metrics(y_test, y_proba, threshold=threshold)

        fold_rows.append({
            "split_id": split["split_id"],
            "test_block_ids": split["test_block_ids"],
            "n_train": split["n_train"],
            "n_test": split["n_test"],
            "embargo_size": split["embargo_size"],
            "feature_method": feature_processing_config["feature_method"],
            "n_original_features": processing_info.get("n_original_features", np.nan),
            "n_processed_features": processing_info.get("n_processed_features", np.nan),
            "n_dropped_features": processing_info.get("n_dropped_features", np.nan),
            "n_pca_components": processing_info.get("n_pca_components", np.nan),
            "pca_explained_variance": processing_info.get("pca_explained_variance", np.nan),
            "trade_count": int((y_pred == 1).sum()),
            "trade_sharpe": per_trade_sharpe(accepted_returns),
            **metrics,
        })

        fold_predictions = fold_test[[DATE_COL, INSTRUMENT_COL, "primary_signal", TARGET_COL, "signed_touch_return"]].copy()
        fold_predictions = fold_predictions.rename(columns={TARGET_COL: "y_true"})
        fold_predictions["split_id"] = split["split_id"]
        fold_predictions["block_id"] = [split["test_block_by_index"][int(idx)] for idx in fold_test.index]
        fold_predictions["feature_method"] = feature_processing_config["feature_method"]
        fold_predictions["model_name"] = rf_config["model_name"]
        fold_predictions["y_proba"] = y_proba
        fold_predictions["y_pred"] = y_pred
        prediction_rows.append(fold_predictions)

    return pd.DataFrame(fold_rows), pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()


def summarise_fold_metrics(fold_metrics_df, ticker, tb_config_name, rf_config, feature_processing_config, num_days):
    params = rf_config["model_params"]
    return {
        "ticker": ticker,
        "tb_config_name": tb_config_name,
        "model_name": rf_config["model_name"],
        "model_params": params,
        "feature_method": feature_processing_config["feature_method"],
        "feature_params": feature_processing_config.get("feature_params", {}),
        "num_days": num_days,
        "n_folds": len(fold_metrics_df),
        "mean_auc": fold_metrics_df["auc"].mean(),
        "median_auc": fold_metrics_df["auc"].median(),
        "std_auc": fold_metrics_df["auc"].std(),
        "mean_accuracy": fold_metrics_df["accuracy"].mean(),
        "mean_precision": fold_metrics_df["precision"].mean(),
        "mean_recall": fold_metrics_df["recall"].mean(),
        "mean_f1": fold_metrics_df["f1"].mean(),
        "median_trade_sharpe": fold_metrics_df["trade_sharpe"].median(),
        "mean_trade_sharpe": fold_metrics_df["trade_sharpe"].mean(),
        "mean_trade_count": fold_metrics_df["trade_count"].mean(),
        "mean_n_train": fold_metrics_df["n_train"].mean(),
        "mean_n_test": fold_metrics_df["n_test"].mean(),
        "mean_original_features": fold_metrics_df["n_original_features"].mean(),
        "mean_processed_features": fold_metrics_df["n_processed_features"].mean(),
        "mean_dropped_features": fold_metrics_df["n_dropped_features"].mean(),
        "mean_pca_components": fold_metrics_df["n_pca_components"].mean(),
        "mean_pca_explained_variance": fold_metrics_df["pca_explained_variance"].mean(),
        "n_estimators": params.get("n_estimators", np.nan),
        "max_depth": params.get("max_depth", np.nan),
        "min_samples_leaf": params.get("min_samples_leaf", np.nan),
        "max_features": params.get("max_features", np.nan),
        "class_weight": params.get("class_weight", np.nan),
    }


def run_rf_cpcv_grid(datasets_by_config, rf_configs, feature_processing_configs):
    summary_rows = []
    detailed_results = {}
    total_runs = len(datasets_by_config) * len(rf_configs) * len(feature_processing_configs)
    run_counter = 0

    for dataset_key_value, data in datasets_by_config.items():
        ticker = data["ticker"]
        tb_config_name = data["config_name"]
        train_df = data["train_df"]
        feature_cols = data["feature_cols"]
        num_days = data["num_days"]

        for rf_config in rf_configs:
            for feature_processing_config in feature_processing_configs:
                run_counter += 1
                print("=" * 100)
                print(f"Run {run_counter}/{total_runs}")
                print(f"Ticker: {ticker}")
                print(f"TB config: {tb_config_name}")
                print(f"RF config: {rf_config['model_name']}")
                print(f"Feature method: {feature_processing_config['feature_method']}")
                print(f"Train rows: {len(train_df)} | Features: {len(feature_cols)} | Embargo: {num_days}")

                fold_metrics_df, oof_predictions_df = run_cpcv_for_rf_config(
                    rf_config=rf_config,
                    train_df=train_df,
                    feature_cols=feature_cols,
                    feature_processing_config=feature_processing_config,
                    n_blocks=CPCV_N_BLOCKS,
                    n_test_blocks=CPCV_N_TEST_BLOCKS,
                    embargo_size=num_days,
                    threshold=0.5,
                )

                result_key = f"{ticker}__{tb_config_name}__{rf_config['model_name']}__{feature_processing_config['feature_method']}"
                detailed_results[result_key] = {
                    "dataset_key": dataset_key_value,
                    "ticker": ticker,
                    "tb_config_name": tb_config_name,
                    "rf_config": rf_config,
                    "feature_processing_config": feature_processing_config,
                    "fold_metrics": fold_metrics_df,
                    "oof_predictions": oof_predictions_df,
                }
                summary = summarise_fold_metrics(fold_metrics_df, ticker, tb_config_name, rf_config, feature_processing_config, num_days)
                summary_rows.append(summary)
                print(f"Mean AUC: {summary['mean_auc']:.4f} | Median Sharpe: {summary['median_trade_sharpe']:.4f}")

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["mean_auc", "std_auc", "median_trade_sharpe"],
        ascending=[False, True, False],
    ).reset_index(drop=True)
    return summary_df, detailed_results


rf_cpcv_summary, rf_detailed_results = run_rf_cpcv_grid(
    datasets_by_config=datasets_by_config,
    rf_configs=RF_CONFIGS,
    feature_processing_configs=FEATURE_PROCESSING_CONFIGS,
)

display(rf_cpcv_summary.head(20))


Run 1/96
Ticker: cl1s
TB config: ewma_10d_tp1_5_sl1_5
RF config: rf_shallow
Feature method: corr_cluster
Train rows: 333 | Features: 145 | Embargo: 10
Mean AUC: 0.5501 | Median Sharpe: 0.3045
Run 2/96
Ticker: cl1s
TB config: ewma_10d_tp1_5_sl1_5
RF config: rf_shallow
Feature method: pca
Train rows: 333 | Features: 145 | Embargo: 10
Mean AUC: 0.4284 | Median Sharpe: 0.2144
Run 3/96
Ticker: cl1s
TB config: ewma_10d_tp1_5_sl1_5
RF config: rf_medium_depth
Feature method: corr_cluster
Train rows: 333 | Features: 145 | Embargo: 10
Mean AUC: 0.5446 | Median Sharpe: 0.3034
Run 4/96
Ticker: cl1s
TB config: ewma_10d_tp1_5_sl1_5
RF config: rf_medium_depth
Feature method: pca
Train rows: 333 | Features: 145 | Embargo: 10
Mean AUC: 0.4259 | Median Sharpe: 0.2248
Run 5/96
Ticker: cl1s
TB config: ewma_10d_tp2_sl2
RF config: rf_shallow
Feature method: corr_cluster
Train rows: 333 | Features: 145 | Embargo: 10
Mean AUC: 0.5887 | Median Sharpe: 0.3733
Run 6/96
Ticker: cl1s
TB config: ewma_10d_tp2_sl2
RF

,ticker,tb_config_name,model_name,model_params,feature_method,feature_params,num_days,n_folds,mean_auc,median_auc,...,mean_original_features,mean_processed_features,mean_dropped_features,mean_pca_components,mean_pca_explained_variance,n_estimators,max_depth,min_samples_leaf,max_features,class_weight
0,ho1s,ewma_5d_tp2_sl2,rf_medium_depth,"{'n_estimators': 500, 'max_depth': 8, 'min_sam...",pca,"{'n_components': 0.95, 'standardize': True, 'i...",5,28,0.659588,0.672727,...,145.0,13.785714,0.000000,12.785714,0.955324,500,8,8,sqrt,balanced
1,ho1s,ewma_5d_tp2_sl2,rf_shallow,"{'n_estimators': 300, 'max_depth': 4, 'min_sam...",pca,"{'n_components': 0.95, 'standardize': True, 'i...",5,28,0.643674,0.653846,...,145.0,13.785714,0.000000,12.785714,0.955324,300,4,10,sqrt,balanced
2,cl1s,parkinson_10d_tp2_sl2,rf_shallow,"{'n_estimators': 300, 'max_depth': 4, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",10,28,0.643080,0.666888,...,145.0,67.250000,77.750000,NaN,NaN,300,4,10,sqrt,balanced
3,cl1s,garman_klass_10d_tp2_sl2,rf_shallow,"{'n_estimators': 300, 'max_depth': 4, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",10,28,0.643080,0.666888,...,145.0,67.250000,77.750000,NaN,NaN,300,4,10,sqrt,balanced
4,cl1s,parkinson_10d_tp2_sl2,rf_medium_depth,"{'n_estimators': 500, 'max_depth': 8, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",10,28,0.635559,0.677468,...,145.0,67.250000,77.750000,NaN,NaN,500,8,8,sqrt,balanced
5,cl1s,garman_klass_10d_tp2_sl2,rf_medium_depth,"{'n_estimators': 500, 'max_depth': 8, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",10,28,0.635559,0.677468,...,145.0,67.250000,77.750000,NaN,NaN,500,8,8,sqrt,balanced
6,ho1s,ewma_10d_tp1_5_sl1_5,rf_medium_depth,"{'n_estimators': 500, 'max_depth': 8, 'min_sam...",pca,"{'n_components': 0.95, 'standardize': True, 'i...",10,28,0.622955,0.647727,...,145.0,12.142857,0.000000,11.142857,0.954814,500,8,8,sqrt,balanced
7,ho1s,ewma_5d_tp2_sl2,rf_shallow,"{'n_estimators': 300, 'max_depth': 4, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",5,28,0.617609,0.604167,...,145.0,60.214286,84.785714,NaN,NaN,300,4,10,sqrt,balanced
8,cl1s,ewma_5d_tp2_sl2,rf_medium_depth,"{'n_estimators': 500, 'max_depth': 8, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",5,28,0.614599,0.611719,...,145.0,67.750000,77.250000,NaN,NaN,500,8,8,sqrt,balanced
9,cl1s,rolling_10d_tp2_sl2,rf_shallow,"{'n_estimators': 300, 'max_depth': 4, 'min_sam...",corr_cluster,"{'corr_threshold': 0.9, 'corr_method': 'spearm...",10,28,0.614242,0.642744,...,145.0,67.250000,77.750000,NaN,NaN,300,4,10,sqrt,balanced


## 9. Universal Ranking And Stitched CPCV Path Sharpe

In [16]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def n_choose_k(n: int, k: int) -> int:
    if k < 0 or k > n:
        return 0
    return int(__import__("math").comb(n, k))


def add_cpcv_path_ids(oof_predictions_df: pd.DataFrame, n_blocks: int = CPCV_N_BLOCKS, n_test_blocks: int = CPCV_N_TEST_BLOCKS) -> pd.DataFrame:
    required = {"split_id", "block_id", DATE_COL, "y_pred", "signed_touch_return"}
    missing = required - set(oof_predictions_df.columns)
    if missing:
        raise ValueError(f"OOF predictions missing columns needed for CPCV paths: {sorted(missing)}")

    expected_paths = n_choose_k(n_blocks - 1, n_test_blocks - 1)
    split_block_pairs = (
        oof_predictions_df[["split_id", "block_id"]]
        .drop_duplicates()
        .sort_values(["split_id", "block_id"])
        .reset_index(drop=True)
    )
    block_seen_counts = {block_id: 0 for block_id in range(n_blocks)}
    mapping_rows = []
    for row in split_block_pairs.itertuples(index=False):
        block_id = int(row.block_id)
        path_id = block_seen_counts[block_id]
        mapping_rows.append({"split_id": int(row.split_id), "block_id": block_id, "cpcv_path_id": path_id})
        block_seen_counts[block_id] += 1

    bad_counts = {block: count for block, count in block_seen_counts.items() if count != expected_paths}
    if bad_counts:
        raise ValueError(f"CPCV path stitching expected {expected_paths} appearances per block, got {bad_counts}.")

    mapping_df = pd.DataFrame(mapping_rows)
    stitched = oof_predictions_df.merge(mapping_df, on=["split_id", "block_id"], how="left")
    if stitched["cpcv_path_id"].isna().any():
        raise ValueError("Some OOF rows could not be assigned to a CPCV path.")
    stitched["cpcv_path_id"] = stitched["cpcv_path_id"].astype(int)

    path_block_counts = stitched.groupby("cpcv_path_id")["block_id"].nunique()
    if not path_block_counts.eq(n_blocks).all():
        raise ValueError(f"Each CPCV path should contain all {n_blocks} blocks. Got: {path_block_counts.to_dict()}")

    return stitched.sort_values(["cpcv_path_id", DATE_COL]).reset_index(drop=True)


def compute_cpcv_path_sharpes(oof_predictions_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    stitched = add_cpcv_path_ids(oof_predictions_df)
    path_rows = []
    for path_id, path_df in stitched.groupby("cpcv_path_id"):
        accepted_returns = path_df.loc[path_df["y_pred"] == 1, "signed_touch_return"]
        path_rows.append({
            "cpcv_path_id": int(path_id),
            "n_blocks_in_path": int(path_df["block_id"].nunique()),
            "n_rows": int(len(path_df)),
            "trade_count": int((path_df["y_pred"] == 1).sum()),
            "path_trade_sharpe": per_trade_sharpe(accepted_returns),
            "mean_trade_return": accepted_returns.mean(),
        })
    path_metrics = pd.DataFrame(path_rows)
    return stitched, path_metrics


path_summary_rows = []
for row in rf_cpcv_summary.itertuples(index=False):
    key = f"{row.ticker}__{row.tb_config_name}__{row.model_name}__{row.feature_method}"
    oof_predictions = rf_detailed_results[key]["oof_predictions"].copy()
    stitched_oof, path_metrics = compute_cpcv_path_sharpes(oof_predictions)
    rf_detailed_results[key]["stitched_oof_predictions"] = stitched_oof
    rf_detailed_results[key]["path_metrics"] = path_metrics
    path_summary_rows.append({
        "result_key": key,
        "n_cpcv_paths": int(path_metrics["cpcv_path_id"].nunique()),
        "median_path_trade_sharpe": path_metrics["path_trade_sharpe"].median(),
        "mean_path_trade_sharpe": path_metrics["path_trade_sharpe"].mean(),
        "path_sharpe_iqr": path_metrics["path_trade_sharpe"].quantile(0.75) - path_metrics["path_trade_sharpe"].quantile(0.25),
        "min_path_trade_sharpe": path_metrics["path_trade_sharpe"].min(),
        "max_path_trade_sharpe": path_metrics["path_trade_sharpe"].max(),
    })

path_summary_df = pd.DataFrame(path_summary_rows)
rf_cpcv_summary["result_key"] = rf_cpcv_summary.apply(
    lambda row: f"{row['ticker']}__{row['tb_config_name']}__{row['model_name']}__{row['feature_method']}",
    axis=1,
)
rf_cpcv_summary = rf_cpcv_summary.merge(path_summary_df, on="result_key", how="left")
rf_cpcv_summary = rf_cpcv_summary.sort_values(
    ["mean_auc", "std_auc", "median_path_trade_sharpe", "path_sharpe_iqr"],
    ascending=[False, True, False, True],
).reset_index(drop=True)

UNIVERSAL_MODEL_COMPARISON_COLUMNS = [
    "model_type",
    "ticker",
    "tb_config_name",
    "feature_method",
    "mean_auc",
    "std_auc",
    "median_auc",
    "n_folds",
    "n_cpcv_paths",
    "median_path_trade_sharpe",
    "mean_path_trade_sharpe",
    "path_sharpe_iqr",
    "min_path_trade_sharpe",
    "max_path_trade_sharpe",
    "mean_precision",
    "mean_recall",
    "mean_f1",
    "mean_trade_count",
    "num_days",
    "model_name",
    "penalty",
    "C",
    "l1_ratio",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "class_weight",
    "feature_params",
]


def build_universal_model_comparison(summary_df: pd.DataFrame, model_type: str = "random_forest") -> pd.DataFrame:
    comparison_df = summary_df.copy()
    comparison_df.insert(0, "model_type", model_type)
    for col in ["penalty", "C", "l1_ratio"]:
        if col not in comparison_df.columns:
            comparison_df[col] = np.nan
    if "feature_params" in comparison_df.columns:
        comparison_df["feature_params"] = comparison_df["feature_params"].apply(
            lambda value: json.dumps(value) if isinstance(value, dict) else value
        )
    missing_cols = [col for col in UNIVERSAL_MODEL_COMPARISON_COLUMNS if col not in comparison_df.columns]
    for col in missing_cols:
        comparison_df[col] = np.nan
    return comparison_df[UNIVERSAL_MODEL_COMPARISON_COLUMNS].sort_values(
        ["mean_auc", "std_auc", "median_path_trade_sharpe", "path_sharpe_iqr"],
        ascending=[False, True, False, True],
    ).reset_index(drop=True)


rf_model_comparison_df = build_universal_model_comparison(rf_cpcv_summary)
top5_by_auc = (
    rf_model_comparison_df
    .sort_values(["ticker", "mean_auc", "std_auc", "median_path_trade_sharpe", "path_sharpe_iqr"], ascending=[True, False, True, False, True])
    .groupby("ticker", as_index=False, group_keys=False)
    .head(5)
    .reset_index(drop=True)
)
top3_by_auc = rf_cpcv_summary.head(3).copy()

top5_by_auc.to_csv(RF_RESULTS_PATH, index=False)
print("Saved RF top-5-by-AUC-per-commodity results to:", RF_RESULTS_PATH)
print(
    f"CPCV setup: {CPCV_N_BLOCKS} blocks, {CPCV_N_TEST_BLOCKS} test blocks per model -> "
    f"{n_choose_k(CPCV_N_BLOCKS, CPCV_N_TEST_BLOCKS)} fitted CPCV models and "
    f"{n_choose_k(CPCV_N_BLOCKS - 1, CPCV_N_TEST_BLOCKS - 1)} stitched paths."
)
print("Top 5 RF candidates per commodity:")
display(top5_by_auc)


KeyError: 'median_path_trade_sharpe'

## 10. Select Best Candidate Per Ticker And Estimate OOF Thresholds

In [21]:
def average_oof_predictions(oof_predictions_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [DATE_COL]

    if INSTRUMENT_COL in oof_predictions_df.columns:
        group_cols.append(INSTRUMENT_COL)

    required_cols = group_cols + [
        "primary_signal",
        "y_true",
        "y_proba",
        "signed_touch_return",
    ]

    missing_cols = [col for col in required_cols if col not in oof_predictions_df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns in OOF predictions: {missing_cols}")

    return (
        oof_predictions_df
        .groupby(group_cols, as_index=False)
        .agg(
            primary_signal=("primary_signal", "first"),
            y_true=("y_true", "first"),
            y_proba=("y_proba", "mean"),
            signed_touch_return=("signed_touch_return", "first"),
            n_oof_predictions=("y_proba", "size"),
        )
    )


def estimate_ev_threshold(train_eval_df: pd.DataFrame, n_bootstrap: int = 2000) -> tuple[dict, pd.DataFrame]:
    rng = np.random.default_rng(RANDOM_STATE)

    required_cols = ["y_true", "signed_touch_return"]
    missing_cols = [col for col in required_cols if col not in train_eval_df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns for EV threshold: {missing_cols}")

    df = train_eval_df[["y_true", "signed_touch_return"]].dropna().copy()
    df["y_true"] = df["y_true"].astype(int)

    good_returns = df.loc[df["y_true"] == 1, "signed_touch_return"].to_numpy()
    bad_returns = df.loc[df["y_true"] == 0, "signed_touch_return"].to_numpy()

    if len(good_returns) == 0 or len(bad_returns) == 0:
        summary = {
            "r_G_mean_return_when_y_1": np.nan,
            "r_L_mean_return_when_y_0": np.nan,
            "raw_threshold": np.nan,
            "clipped_threshold": 0.5,
            "threshold_valid": False,
            "n_good": len(good_returns),
            "n_bad": len(bad_returns),
            "bootstrap_threshold_median": np.nan,
            "bootstrap_threshold_5pct": np.nan,
            "bootstrap_threshold_95pct": np.nan,
        }
        return summary, pd.DataFrame()

    r_G = float(np.mean(good_returns))
    r_L = float(np.mean(bad_returns))

    denominator = r_G - r_L

    if denominator <= 0:
        raw_threshold = np.nan
        clipped_threshold = 0.5
        threshold_valid = False
    else:
        raw_threshold = -r_L / denominator
        clipped_threshold = float(np.clip(raw_threshold, 0.0, 1.0))
        threshold_valid = np.isfinite(raw_threshold)

    bootstrap_rows = []

    for i in range(n_bootstrap):
        boot_good = rng.choice(good_returns, size=len(good_returns), replace=True)
        boot_bad = rng.choice(bad_returns, size=len(bad_returns), replace=True)

        boot_r_G = float(np.mean(boot_good))
        boot_r_L = float(np.mean(boot_bad))

        boot_denom = boot_r_G - boot_r_L
        boot_threshold = -boot_r_L / boot_denom if boot_denom > 0 else np.nan

        bootstrap_rows.append({
            "bootstrap_id": i + 1,
            "r_G": boot_r_G,
            "r_L": boot_r_L,
            "raw_threshold": boot_threshold,
            "clipped_threshold": float(np.clip(boot_threshold, 0.0, 1.0)) if np.isfinite(boot_threshold) else np.nan,
        })

    bootstrap_df = pd.DataFrame(bootstrap_rows)

    summary = {
        "r_G_mean_return_when_y_1": r_G,
        "r_L_mean_return_when_y_0": r_L,
        "raw_threshold": raw_threshold,
        "clipped_threshold": clipped_threshold,
        "threshold_valid": threshold_valid,
        "n_good": len(good_returns),
        "n_bad": len(bad_returns),
        "bootstrap_threshold_median": bootstrap_df["clipped_threshold"].median(),
        "bootstrap_threshold_5pct": bootstrap_df["clipped_threshold"].quantile(0.05),
        "bootstrap_threshold_95pct": bootstrap_df["clipped_threshold"].quantile(0.95),
    }

    return summary, bootstrap_df


def clean_duplicate_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    duplicate_metric_cols = [
        "n_cpcv_paths",
        "median_path_trade_sharpe",
        "mean_path_trade_sharpe",
        "path_sharpe_iqr",
        "min_path_trade_sharpe",
        "max_path_trade_sharpe",
    ]

    for col in duplicate_metric_cols:
        col_x = f"{col}_x"
        col_y = f"{col}_y"

        if col in df.columns:
            continue

        if col_x in df.columns and col_y in df.columns:
            df[col] = df[col_x].combine_first(df[col_y])
        elif col_x in df.columns:
            df[col] = df[col_x]
        elif col_y in df.columns:
            df[col] = df[col_y]

    return df


rf_cpcv_summary_clean = clean_duplicate_metric_columns(rf_cpcv_summary)


best_by_ticker = (
    rf_cpcv_summary_clean
    .sort_values(
        [
            "ticker",
            "mean_auc",
            "std_auc",
            "median_path_trade_sharpe",
            "path_sharpe_iqr",
        ],
        ascending=[
            True,
            False,
            True,
            False,
            True,
        ],
    )
    .groupby("ticker", as_index=False)
    .head(1)
    .reset_index(drop=True)
)


threshold_rows = []
threshold_bootstrap_by_ticker = {}
avg_oof_by_ticker = {}

for row in best_by_ticker.itertuples(index=False):
    result_key = row.result_key

    avg_oof_predictions = average_oof_predictions(
        rf_detailed_results[result_key]["oof_predictions"]
    )

    threshold_summary, threshold_bootstrap_df = estimate_ev_threshold(
        avg_oof_predictions
    )

    threshold_rows.append({
        "ticker": row.ticker,
        "result_key": result_key,
        **threshold_summary,
    })

    threshold_bootstrap_by_ticker[row.ticker] = threshold_bootstrap_df
    avg_oof_by_ticker[row.ticker] = avg_oof_predictions


threshold_summary_df = pd.DataFrame(threshold_rows)

print("Best AUC RF candidate per ticker:")

display_cols = [
    "ticker",
    "tb_config_name",
    "model_name",
    "feature_method",
    "mean_auc",
    "std_auc",
    "n_cpcv_paths",
    "median_path_trade_sharpe",
    "path_sharpe_iqr",
    "result_key",
]

display(best_by_ticker[display_cols])

print("OOF EV thresholds per ticker:")
display(threshold_summary_df)

Best AUC RF candidate per ticker:


,ticker,tb_config_name,model_name,feature_method,mean_auc,std_auc,n_cpcv_paths,median_path_trade_sharpe,path_sharpe_iqr,result_key
0,cl1s,parkinson_10d_tp2_sl2,rf_shallow,corr_cluster,0.643080,0.192026,7,0.421916,0.015622,cl1s__parkinson_10d_tp2_sl2__rf_shallow__corr_...
1,ho1s,ewma_5d_tp2_sl2,rf_medium_depth,pca,0.659588,0.165142,7,0.396755,0.141361,ho1s__ewma_5d_tp2_sl2__rf_medium_depth__pca
2,ng1s,garman_klass_10d_tp2_sl2,rf_shallow,pca,0.586819,0.134154,7,0.308722,0.149032,ng1s__garman_klass_10d_tp2_sl2__rf_shallow__pca
3,rb1s,ewma_10d_tp2_sl2,rf_shallow,corr_cluster,0.507104,0.096825,7,0.030050,0.037797,rb1s__ewma_10d_tp2_sl2__rf_shallow__corr_cluster


OOF EV thresholds per ticker:


,ticker,result_key,r_G_mean_return_when_y_1,r_L_mean_return_when_y_0,raw_threshold,clipped_threshold,threshold_valid,n_good,n_bad,bootstrap_threshold_median,bootstrap_threshold_5pct,bootstrap_threshold_95pct
0,cl1s,cl1s__parkinson_10d_tp2_sl2__rf_shallow__corr_...,0.085465,-0.052273,0.379510,0.379510,True,218,115,0.379937,0.340981,0.420422
1,ho1s,ho1s__ewma_5d_tp2_sl2__rf_medium_depth__pca,0.048001,-0.037553,0.438938,0.438938,True,41,20,0.437845,0.345876,0.525548
2,ng1s,ng1s__garman_klass_10d_tp2_sl2__rf_shallow__pca,0.096027,-0.091460,0.487820,0.487820,True,43,25,0.487378,0.446850,0.526425
3,rb1s,rb1s__ewma_10d_tp2_sl2__rf_shallow__corr_cluster,0.069824,-0.061935,0.470059,0.470059,True,245,259,0.467954,0.438185,0.500810


In [18]:
print(rf_cpcv_summary.columns.tolist())

['ticker', 'tb_config_name', 'model_name', 'model_params', 'feature_method', 'feature_params', 'num_days', 'n_folds', 'mean_auc', 'median_auc', 'std_auc', 'mean_accuracy', 'mean_precision', 'mean_recall', 'mean_f1', 'median_trade_sharpe', 'mean_trade_sharpe', 'mean_trade_count', 'mean_n_train', 'mean_n_test', 'mean_original_features', 'mean_processed_features', 'mean_dropped_features', 'mean_pca_components', 'mean_pca_explained_variance', 'n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 'class_weight', 'result_key', 'n_cpcv_paths_x', 'median_path_trade_sharpe_x', 'mean_path_trade_sharpe_x', 'path_sharpe_iqr_x', 'min_path_trade_sharpe_x', 'max_path_trade_sharpe_x', 'n_cpcv_paths_y', 'median_path_trade_sharpe_y', 'mean_path_trade_sharpe_y', 'path_sharpe_iqr_y', 'min_path_trade_sharpe_y', 'max_path_trade_sharpe_y']


## 11. Final Fit And OOS Evaluation Per Ticker

In [22]:
def fit_final_rf_for_result(result_key: str, ev_threshold: float) -> dict:
    detail = rf_detailed_results[result_key]
    ticker = detail["ticker"]
    tb_config_name = detail["tb_config_name"]
    data = datasets_by_config[detail["dataset_key"]]
    rf_config = detail["rf_config"]
    feature_processing_config = detail["feature_processing_config"]

    final_train_df = data["train_df"].copy()
    final_test_df = data["test_df"].copy()
    if final_test_df.empty:
        raise ValueError(f"OOS test set is empty for {ticker}/{tb_config_name}.")

    y_final_train = final_train_df[TARGET_COL].astype(int)
    X_final_processed, X_oos_processed, final_processed_feature_cols, final_processing_info = apply_feature_processing_for_fold(
        fold_train=final_train_df,
        fold_test=final_test_df,
        feature_cols=data["feature_cols"],
        y_fold_train=y_final_train,
        feature_processing_config=feature_processing_config,
    )

    final_rf_model = make_random_forest(rf_config["model_params"])
    final_rf_model.fit(X_final_processed, y_final_train)
    oos_proba = get_positive_class_proba(final_rf_model, X_oos_processed)

    oos_predictions_df = final_test_df[[DATE_COL, INSTRUMENT_COL, "primary_signal", TARGET_COL, "signed_touch_return"]].copy()
    oos_predictions_df = oos_predictions_df.rename(columns={TARGET_COL: "y_true"})
    oos_predictions_df["ticker"] = ticker
    oos_predictions_df["tb_config_name"] = tb_config_name
    oos_predictions_df["model_name"] = rf_config["model_name"]
    oos_predictions_df["feature_method"] = feature_processing_config["feature_method"]
    oos_predictions_df["y_proba"] = oos_proba
    oos_predictions_df["ev_threshold"] = ev_threshold
    oos_predictions_df["y_pred_0_5"] = (oos_predictions_df["y_proba"] >= 0.5).astype(int)
    oos_predictions_df["y_pred_ev"] = (oos_predictions_df["y_proba"] >= ev_threshold).astype(int)

    return {
        "ticker": ticker,
        "tb_config_name": tb_config_name,
        "result_key": result_key,
        "rf_config": rf_config,
        "feature_processing_config": feature_processing_config,
        "data": data,
        "final_train_df": final_train_df,
        "final_test_df": final_test_df,
        "final_rf_model": final_rf_model,
        "final_processing_info": final_processing_info,
        "final_processed_feature_cols": final_processed_feature_cols,
        "oos_predictions_df": oos_predictions_df,
        "oos_metrics_05": classification_metrics(oos_predictions_df["y_true"], oos_predictions_df["y_proba"], threshold=0.5),
        "oos_metrics_ev": classification_metrics(oos_predictions_df["y_true"], oos_predictions_df["y_proba"], threshold=ev_threshold),
    }


final_results_by_ticker = {}
oos_metric_rows = []
for row in best_by_ticker.itertuples(index=False):
    threshold_row = threshold_summary_df.loc[threshold_summary_df["ticker"] == row.ticker].iloc[0]
    ev_threshold = float(threshold_row["clipped_threshold"])
    result = fit_final_rf_for_result(row.result_key, ev_threshold)
    final_results_by_ticker[row.ticker] = result
    oos_metric_rows.append({"ticker": row.ticker, "threshold": "0.5", **result["oos_metrics_05"]})
    oos_metric_rows.append({"ticker": row.ticker, "threshold": "ev", "ev_threshold": ev_threshold, **result["oos_metrics_ev"]})

oos_metrics_df = pd.DataFrame(oos_metric_rows)
print("OOS metrics per ticker:")
display(oos_metrics_df)


OOS metrics per ticker:


,ticker,threshold,accuracy,precision,recall,f1,auc,tn,fp,fn,tp,ev_threshold
0,cl1s,0.5,0.794872,0.805195,0.984127,0.885714,0.637037,0,15,1,62,NaN
1,cl1s,ev,0.794872,0.805195,0.984127,0.885714,0.637037,0,15,1,62,0.379510
2,ho1s,0.5,0.000000,0.000000,0.000000,0.000000,NaN,0,2,0,0,NaN
3,ho1s,ev,0.000000,0.000000,0.000000,0.000000,NaN,0,2,0,0,0.438938
4,ng1s,0.5,0.461538,0.404762,0.850000,0.548387,0.425000,7,25,3,17,NaN
5,ng1s,ev,0.461538,0.404762,0.850000,0.548387,0.425000,7,25,3,17,0.487820
6,rb1s,0.5,0.684211,0.913043,0.567568,0.700000,0.758446,36,4,32,42,NaN
7,rb1s,ev,0.710526,0.886792,0.635135,0.740157,0.758446,34,6,27,47,0.470059


## 12. Strategy Diagnostics Per Ticker

In [23]:
def build_strategy_analysis(predictions_df: pd.DataFrame, horizon_days: int, threshold: float):
    strategy_df = predictions_df.copy().dropna(subset=["signed_touch_return"]).sort_values(DATE_COL).reset_index(drop=True)
    strategy_df["meta_take_trade"] = (strategy_df["y_proba"] >= threshold).astype(int)
    strategy_df["primary_return"] = strategy_df["signed_touch_return"]
    strategy_df["meta_return"] = strategy_df["meta_take_trade"] * strategy_df["signed_touch_return"]
    strategy_df["primary_nav"] = (1.0 + strategy_df["primary_return"]).cumprod()
    strategy_df["meta_nav"] = (1.0 + strategy_df["meta_return"]).cumprod()

    periods_per_year = 252 / horizon_days
    summary_df = pd.DataFrame([
        {
            "strategy": "Primary alone",
            "n_trades_taken": int(len(strategy_df)),
            "trade_rate": 1.0,
            "mean_event_return": strategy_df["primary_return"].mean(),
            "total_profit_pct": strategy_df["primary_nav"].iloc[-1] - 1.0,
            "annualised_sharpe": annualised_sharpe(strategy_df["primary_return"], periods_per_year),
            "max_drawdown": max_drawdown(strategy_df["primary_nav"]),
        },
        {
            "strategy": "Primary + RF meta filter",
            "n_trades_taken": int(strategy_df["meta_take_trade"].sum()),
            "trade_rate": float(strategy_df["meta_take_trade"].mean()),
            "mean_event_return": strategy_df["meta_return"].mean(),
            "total_profit_pct": strategy_df["meta_nav"].iloc[-1] - 1.0,
            "annualised_sharpe": annualised_sharpe(strategy_df["meta_return"], periods_per_year),
            "max_drawdown": max_drawdown(strategy_df["meta_nav"]),
        },
    ])
    return strategy_df, summary_df


strategy_summary_tables = []
for ticker, result in final_results_by_ticker.items():
    threshold = float(result["oos_predictions_df"]["ev_threshold"].iloc[0])
    strategy_df, strategy_summary_df = build_strategy_analysis(result["oos_predictions_df"], result["data"]["num_days"], threshold)
    strategy_df["ticker"] = ticker
    strategy_summary_df.insert(0, "ticker", ticker)
    result["oos_strategy_df"] = strategy_df
    result["oos_strategy_summary_df"] = strategy_summary_df
    strategy_summary_tables.append(strategy_summary_df)

oos_strategy_summary_all_df = pd.concat(strategy_summary_tables, ignore_index=True)
print("OOS strategy comparison per ticker:")
display(oos_strategy_summary_all_df)


OOS strategy comparison per ticker:


,ticker,strategy,n_trades_taken,trade_rate,mean_event_return,total_profit_pct,annualised_sharpe,max_drawdown
0,cl1s,Primary alone,78,1.000000,0.037451,14.129295,2.961454,-0.519164
1,cl1s,Primary + RF meta filter,77,0.987179,0.036421,13.004218,2.882422,-0.519164
2,ho1s,Primary alone,2,1.000000,-0.067079,-0.130086,-16.289481,-0.046407
3,ho1s,Primary + RF meta filter,2,1.000000,-0.067079,-0.130086,-16.289481,-0.046407
4,ng1s,Primary alone,52,1.000000,-0.020341,-0.773888,-0.802439,-0.960158
5,ng1s,Primary + RF meta filter,42,0.807692,-0.007476,-0.504412,-0.337660,-0.875540
6,rb1s,Primary alone,114,1.000000,0.017344,4.648021,1.361361,-0.764387
7,rb1s,Primary + RF meta filter,53,0.464912,0.021752,9.798525,2.957203,-0.173187


## 13. Feature And Cluster Importance Per Ticker

In [24]:
def build_rf_feature_importance(result: dict) -> pd.DataFrame:
    feature_importance_df = pd.DataFrame({
        "feature": result["final_processed_feature_cols"],
        "importance": result["final_rf_model"].feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    feature_importance_df["importance_type"] = "random_forest_mdi"
    return feature_importance_df


def build_cluster_importance_df(feature_importance_df: pd.DataFrame, processing_info: dict) -> pd.DataFrame:
    method = processing_info.get("feature_method", "none")
    importance_df = feature_importance_df.copy()
    importance_df["importance_group"] = importance_df["feature"]
    importance_df["group_type"] = "single_feature"
    importance_df["cluster_id"] = np.nan
    importance_df["cluster_n_features"] = 1
    importance_df["cluster_features"] = importance_df["feature"].apply(lambda x: [x])
    importance_df["dropped_cluster_features"] = [[] for _ in range(len(importance_df))]

    if method == "corr_cluster":
        selector = processing_info.get("processor")
        if selector is not None and hasattr(selector, "cluster_summary_"):
            cluster_summary = selector.cluster_summary_.copy()
            if not cluster_summary.empty:
                rep_to_group = {}
                rep_to_cluster_id = {}
                rep_to_n_features = {}
                rep_to_cluster_features = {}
                rep_to_dropped_features = {}
                for row in cluster_summary.itertuples(index=False):
                    cluster_id = int(row.cluster_id)
                    representative = row.representative_feature
                    cluster_features = list(row.cluster_features)
                    dropped_features = list(row.dropped_features)
                    rep_to_group[representative] = f"cluster_{cluster_id:03d}: {representative} (rep, n={len(cluster_features)})"
                    rep_to_cluster_id[representative] = cluster_id
                    rep_to_n_features[representative] = len(cluster_features)
                    rep_to_cluster_features[representative] = cluster_features
                    rep_to_dropped_features[representative] = dropped_features
                is_cluster_rep = importance_df["feature"].isin(rep_to_group.keys())
                importance_df.loc[is_cluster_rep, "importance_group"] = importance_df.loc[is_cluster_rep, "feature"].map(rep_to_group)
                importance_df.loc[is_cluster_rep, "group_type"] = "correlation_cluster"
                importance_df.loc[is_cluster_rep, "cluster_id"] = importance_df.loc[is_cluster_rep, "feature"].map(rep_to_cluster_id)
                importance_df.loc[is_cluster_rep, "cluster_n_features"] = importance_df.loc[is_cluster_rep, "feature"].map(rep_to_n_features)
                importance_df.loc[is_cluster_rep, "cluster_features"] = importance_df.loc[is_cluster_rep, "feature"].map(rep_to_cluster_features)
                importance_df.loc[is_cluster_rep, "dropped_cluster_features"] = importance_df.loc[is_cluster_rep, "feature"].map(rep_to_dropped_features)
    elif method == "pca":
        importance_df["group_type"] = np.where(importance_df["feature"].str.startswith("pca_"), "pca_component", "single_feature")

    cluster_df = (
        importance_df
        .groupby("importance_group", as_index=False)
        .agg(
            cluster_importance=("importance", "sum"),
            group_type=("group_type", "first"),
            representative_model_features=("feature", lambda x: list(x)),
            n_model_features=("feature", "count"),
            cluster_id=("cluster_id", "first"),
            cluster_n_features=("cluster_n_features", "first"),
            cluster_features=("cluster_features", "first"),
            dropped_cluster_features=("dropped_cluster_features", "first"),
        )
        .sort_values("cluster_importance", ascending=False)
        .reset_index(drop=True)
    )
    cluster_df["importance_type"] = "random_forest_mdi"
    return cluster_df


feature_importance_tables = []
cluster_importance_tables = []
for ticker, result in final_results_by_ticker.items():
    feature_importance_df = build_rf_feature_importance(result)
    cluster_importance_df = build_cluster_importance_df(feature_importance_df, result["final_processing_info"])
    feature_importance_df.insert(0, "ticker", ticker)
    cluster_importance_df.insert(0, "ticker", ticker)
    result["feature_importance_df"] = feature_importance_df
    result["cluster_importance_df"] = cluster_importance_df
    feature_importance_tables.append(feature_importance_df)
    cluster_importance_tables.append(cluster_importance_df)

feature_importance_all_df = pd.concat(feature_importance_tables, ignore_index=True)
cluster_importance_all_df = pd.concat(cluster_importance_tables, ignore_index=True)
print("Top RF feature importance per ticker:")
display(feature_importance_all_df.groupby("ticker").head(10))
print("Top RF cluster/component importance per ticker:")
display(cluster_importance_all_df.groupby("ticker").head(10))


Top RF feature importance per ticker:


,ticker,feature,importance,importance_type
0,cl1s,vol_ratio_63_126d,0.163260,random_forest_mdi
1,cl1s,vol_63d,0.096892,random_forest_mdi
2,cl1s,sma50_vs_sma100,0.071630,random_forest_mdi
3,cl1s,bb_lower_dist,0.036184,random_forest_mdi
4,cl1s,atr_20d_pct,0.035179,random_forest_mdi
5,cl1s,bb_width,0.034641,random_forest_mdi
6,cl1s,sma100_slope,0.032491,random_forest_mdi
7,cl1s,ret_spread_20_63,0.027439,random_forest_mdi
8,cl1s,atr_20d,0.024453,random_forest_mdi
9,cl1s,ret_126d,0.023558,random_forest_mdi


Top RF cluster/component importance per ticker:


,ticker,importance_group,cluster_importance,group_type,representative_model_features,n_model_features,cluster_id,cluster_n_features,cluster_features,dropped_cluster_features,importance_type
0,cl1s,vol_ratio_63_126d,0.163260,single_feature,[vol_ratio_63_126d],1,NaN,1,[vol_ratio_63_126d],[],random_forest_mdi
1,cl1s,"cluster_008: vol_63d (rep, n=4)",0.096892,correlation_cluster,[vol_63d],1,8.0,4,"[vol_63d, ewma_vol_63d, park_vol_63d, gk_vol_63d]","[ewma_vol_63d, park_vol_63d, gk_vol_63d]",random_forest_mdi
2,cl1s,sma50_vs_sma100,0.071630,single_feature,[sma50_vs_sma100],1,NaN,1,[sma50_vs_sma100],[],random_forest_mdi
3,cl1s,bb_lower_dist,0.036184,single_feature,[bb_lower_dist],1,NaN,1,[bb_lower_dist],[],random_forest_mdi
4,cl1s,"cluster_007: atr_20d_pct (rep, n=18)",0.035179,correlation_cluster,[atr_20d_pct],1,7.0,18,"[vol_5d, vol_10d, vol_20d, ewma_vol_5d, ewma_v...","[vol_5d, vol_10d, vol_20d, ewma_vol_5d, ewma_v...",random_forest_mdi
5,cl1s,bb_width,0.034641,single_feature,[bb_width],1,NaN,1,[bb_width],[],random_forest_mdi
6,cl1s,sma100_slope,0.032491,single_feature,[sma100_slope],1,NaN,1,[sma100_slope],[],random_forest_mdi
7,cl1s,ret_spread_20_63,0.027439,single_feature,[ret_spread_20_63],1,NaN,1,[ret_spread_20_63],[],random_forest_mdi
8,cl1s,"cluster_010: atr_20d (rep, n=4)",0.024453,correlation_cluster,[atr_20d],1,10.0,4,"[atr_5d, atr_10d, atr_14d, atr_20d]","[atr_5d, atr_10d, atr_14d]",random_forest_mdi
9,cl1s,ret_126d,0.023558,single_feature,[ret_126d],1,NaN,1,[ret_126d],[],random_forest_mdi


## 14. Save Universal Results And RF-Style Outputs

In [25]:
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

top5_by_auc.to_csv(RF_RESULTS_PATH, index=False)

# Preserve existing non-RF rows in the universal model comparison file, then add/replace RF rows.
if MODEL_COMPARISON_RESULTS_PATH.exists():
    existing_comparison = pd.read_csv(MODEL_COMPARISON_RESULTS_PATH)
    if "model_type" in existing_comparison.columns:
        existing_comparison = existing_comparison[existing_comparison["model_type"] != "random_forest"].copy()
    else:
        existing_comparison = pd.DataFrame(columns=UNIVERSAL_MODEL_COMPARISON_COLUMNS)
else:
    existing_comparison = pd.DataFrame(columns=UNIVERSAL_MODEL_COMPARISON_COLUMNS)

combined_model_comparison = pd.concat([existing_comparison, top5_by_auc], ignore_index=True)
combined_model_comparison = combined_model_comparison[UNIVERSAL_MODEL_COMPARISON_COLUMNS]
combined_model_comparison.to_csv(MODEL_COMPARISON_RESULTS_PATH, index=False)

saved_paths = [RF_RESULTS_PATH, MODEL_COMPARISON_RESULTS_PATH]

for ticker, result in final_results_by_ticker.items():
    ticker_summary = rf_cpcv_summary[rf_cpcv_summary["ticker"] == ticker].copy()

    CPCV_SUMMARY_PATH = MODEL_OUTPUT_DIR / f"{ticker}_rf_cpcv_summary.csv"
    OOS_PREDICTIONS_PATH = MODEL_OUTPUT_DIR / f"{ticker}_rf_oos_predictions.csv"
    OOS_STRATEGY_PATH = MODEL_OUTPUT_DIR / f"{ticker}_rf_oos_strategy_returns.csv"
    OOS_STRATEGY_SUMMARY_PATH = MODEL_OUTPUT_DIR / f"{ticker}_rf_oos_strategy_summary.csv"
    FEATURE_IMPORTANCE_PATH = MODEL_OUTPUT_DIR / f"{ticker}_rf_feature_importance.csv"
    CLUSTER_IMPORTANCE_PATH = MODEL_OUTPUT_DIR / f"{ticker}_rf_cluster_importance.csv"

    ticker_summary.to_csv(CPCV_SUMMARY_PATH, index=False)
    result["oos_predictions_df"].to_csv(OOS_PREDICTIONS_PATH, index=False)
    result["oos_strategy_df"].to_csv(OOS_STRATEGY_PATH, index=False)
    result["oos_strategy_summary_df"].to_csv(OOS_STRATEGY_SUMMARY_PATH, index=False)
    result["feature_importance_df"].to_csv(FEATURE_IMPORTANCE_PATH, index=False)
    result["cluster_importance_df"].to_csv(CLUSTER_IMPORTANCE_PATH, index=False)

    saved_paths.extend([
        CPCV_SUMMARY_PATH,
        OOS_PREDICTIONS_PATH,
        OOS_STRATEGY_PATH,
        OOS_STRATEGY_SUMMARY_PATH,
        FEATURE_IMPORTANCE_PATH,
        CLUSTER_IMPORTANCE_PATH,
    ])

print("Saved Random Forest outputs:")
for path in saved_paths:
    print(path)


Saved Random Forest outputs:
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/results/random_forest.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/results/model_comparison.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest/cl1s_rf_cpcv_summary.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest/cl1s_rf_oos_predictions.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest/cl1s_rf_oos_strategy_returns.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest/cl1s_rf_oos_strategy_summary.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest/cl1s_rf_feature_importance.csv
/Users/faisal/Desktop/systematic-strategies-with-machine-learning-Cw/data/model_outputs/random_forest/cl1s_rf_cluster_